# RSNA EXP-001 fold0 probe: artifact persistence + timing

目的:验证 artifact 持久化链路 + 记录 fold0 真实 wall-clock。
变更(2026-09-10, 本地干跑验证后):
1. 修复 duplicate predict_test/plane_oof_auc 块;
2. 包 torch 依赖类进 if HAS_TORCH: 使本地 --help 可测;
3. @torch.no_grad 装饰器改为函数内 with 上下文;
4. 每折保存 test_pred.npy / test_uid.csv / meta.json;
5. experiment-level config.json + fold_assignment.csv;
6. --resume-dir 支持跨 session 续训;
7. UID-aligned ensemble 加载 + 缺失折容错.

命令:python rsna_v3.py --planes sagittal --epochs 3 --folds 5 --only-fold 0 --batch 8 --n-clips 2 --verbose


In [ ]:
# 提交模式:禁网(enable_internet:false),torch 2.2.0+cu118 + nvidia-cu11 依赖全部离线安装
# 坑1:Kaggle 挂载把文件名中 '+' 去掉(2.2.0cu118=非法 PEP440)→ 复制为合法名
# 坑2:torch 2.2.0+cu118 的 CUDA 库在 nvidia-*-cu11 依赖包中(不在 torch wheel 内),
#     只装 torch 会缺 libcudart(v8/v9 实证)→ 必须显式安装全部 13 个 wheel
# 坑3:/kaggle/working 会被收集为 kernel 输出 → 装完立即删除
import subprocess, sys, os, glob, shutil
wh = sorted(glob.glob('/kaggle/input/**/*.whl', recursive=True))
print('挂载 wheel 共', len(wh), '个')
for w in wh: print('  ', os.path.basename(w))
assert len(wh) >= 13, 'FAIL: 离线 wheel 不足(应 13: torch+torchvision+11 nvidia-cu11)'
fixed = []
for w in wh:
    base = os.path.basename(w)
    dst = os.path.join('/kaggle/working', base.replace('cu118', '+cu118', 1) if 'cu118' in base else base)
    shutil.copy(w, dst)
    fixed.append(dst)
print('安装 wheel:', [os.path.basename(f) for f in fixed])
subprocess.check_call([sys.executable,"-m","pip","uninstall","-y","-q","torch","torchvision","torchaudio"])
# 显式安装全部 wheel(--no-deps --no-index: 无版本解析/无联网)
subprocess.check_call([sys.executable,"-m","pip","install","-q","--no-deps","--no-index"] + fixed)
for f in fixed: os.remove(f)
import numpy, torch, torchvision, sklearn, pandas
print("torch", torch.__version__, "cuda", torch.version.cuda, "| numpy", numpy.__version__)
assert torch.cuda.is_available()
print("GPU:", torch.cuda.get_device_name(0))


In [ ]:
# preflight:test DICOM 布局抽样(防训完才暴露 test 读不到,浪费额度)
# 扫描兼容 1/2/3 层挂载:/kaggle/input/<slug>/、/<slug>/<slug>、/competitions/<slug>;
# 跳过 train_series/test_series 大目录(内部不可能有 test.csv)
import os
base='/kaggle/input'
def _find(b, depth=0, max_depth=3):
    if not os.path.isdir(b): return None
    try: entries = sorted(os.listdir(b))
    except OSError: return None
    for e in entries:
        p = os.path.join(b, e)
        if not os.path.isdir(p): continue
        if e in ('train_series', 'test_series'): continue
        if os.path.isfile(os.path.join(p, 'test.csv')) and os.path.isdir(os.path.join(p, 'test_series')):
            return p
        if depth < max_depth - 1:
            r = _find(p, depth + 1)
            if r: return r
    return None
root = _find(base)
if root is None:
    raise SystemExit('FAIL: 未找到含 test_series+test.csv 的比赛挂载(已扫 /kaggle/input 3 层)')
print('root =', root)
ts=os.path.join(root,'test_series'); n_dcm=0
for st in sorted(os.listdir(ts))[:3]:
    sp=os.path.join(ts,st)
    if not os.path.isdir(sp): continue
    for se in sorted(os.listdir(sp))[:3]:
        sdir=os.path.join(sp,se)
        if not os.path.isdir(sdir): continue
        k=len([f for f in os.listdir(sdir) if f.lower().endswith('.dcm')])
        print(' study', st[:14], 'series', se[:14], 'dcm:', k); n_dcm+=k
print('preflight:', 'OK' if n_dcm>0 else 'FAIL: test DICOM 为空')
assert n_dcm>0


In [ ]:
import base64, os
parts = []
parts.append("...")
path = "/kaggle/working/rsna_v3.py"
with open(path, "w", encoding="utf-8") as f:
    f.write(base64.b64decode(parts[0]).decode("utf-8"))
print(f"wrote rsna_v3.py ({os.path.getsize(path)} bytes)")


In [ ]:
import base64, os
parts = []
parts.append('...')
open("/kaggle/working/rsna_report_labels.py","wb").write(base64.b64decode("".join(parts)))
print("rsna_report_labels.py ->", os.path.getsize("/kaggle/working/rsna_report_labels.py")//1024, "KB")

In [ ]:
import subprocess, sys, os
os.chdir("/kaggle/working")
print("=== 运行 EXP-001 fold0 probe: rsna_v3.py --planes sagittal --epochs 3 --folds 5 --only-fold 0 --batch 8 --n-clips 2 --verbose ===")
r = subprocess.run([sys.executable,"-u","rsna_v3.py",
                    "--planes","sagittal","--epochs","3","--folds","5",
                    "--only-fold","0","--batch","8","--n-clips","2","--verbose"],
                   capture_output=True, text=True)
print(r.stdout)
if r.stderr: print("STDERR:", r.stderr[-2500:])
import pandas as pd
sp="/kaggle/working/submission.csv"
if os.path.isfile(sp):
    sub=pd.read_csv(sp)
    print("[提交文件]", sp, sub.shape)
    print("列:", list(sub.columns))
    print("NaN 总数:", int(sub.isna().sum().sum()))
else:
    print("[警告] submission.csv 未生成!")

# 检查 artifact 落盘
import glob, json
artifact_root = "/kaggle/working/artifacts/exp_001_5fold_sagittal"
patterns = [
    f"{artifact_root}/fold0/test_pred.npy",
    f"{artifact_root}/fold0/test_uid.csv",
    f"{artifact_root}/fold0/meta.json",
    f"{artifact_root}/config.json",
    f"{artifact_root}/fold_assignment.csv",
]
for p in patterns:
    if os.path.exists(p):
        size = os.path.getsize(p)
        print(f"[artifact] OK {p} ({size} bytes)")
    else:
        print(f"[artifact] MISSING {p}")

# 打印 meta.json 时间
meta_path = f"{artifact_root}/fold0/meta.json"
if os.path.isfile(meta_path):
    meta = json.load(open(meta_path, "r", encoding="utf-8"))
    print("[timing] train:", meta.get("fold_train_time_sec"), "inference:", meta.get("fold_inference_time_sec"))
